# Vigenere - ablation runs

One variant per run; set `VARIANT` and re-run. Each writes to its own
checkpoint directory.

| VARIANT | position encoding | period conditioning | passes |
|---|---|---|---|
| `vanilla`   | learned absolute | none | 1 |
| `rope_only` | rotary | none | 1 |
| `pate_la`   | learned absolute | period vector | 2 |

`pate_la` is not expected to converge; it is run with three seeds to show that
the failure is not a single unlucky initialisation. The full model is trained in
`vigenere_train_pate.ipynb`.

In [ ]:
import os, sys
from pathlib import Path

def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if in_colab():
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = Path('/content/drive/MyDrive/cryptanalysis-ai')
    %pip install -q pytorch-lightning torchmetrics tensorboard matplotlib
else:
    REPO_DIR = Path.cwd()
    if REPO_DIR.name == 'notebooks':
        REPO_DIR = REPO_DIR.parent

os.environ['CRYPT_DATA_DIR'] = str(REPO_DIR / 'data')
os.environ['CRYPT_CKPT_DIR'] = str(REPO_DIR / 'checkpoints')
os.environ['CRYPT_LOG_DIR']  = str(REPO_DIR / 'tb_logs')
sys.path.insert(0, str(REPO_DIR / 'src'))
print('repo:', REPO_DIR)


In [ ]:
VARIANT = "vanilla"     # "vanilla" | "rope_only" | "pate_la"
SEED    = 42            # pate_la is run with 42, 1337 and 2024

import importlib
import torch, config
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from pytorch_lightning.loggers import TensorBoardLogger

SPECS = {
    "vanilla":   dict(module="models.vigenere.model_vanilla",
                      ckpt="vigenere_vanilla", two_pass=False),
    "rope_only": dict(module="models.vigenere.model_rope_only",
                      ckpt="vigenere_rope_only", two_pass=False),
    "pate_la":   dict(module="models.vigenere.model_pate_la",
                      ckpt="vigenere_pate_la", two_pass=True),
}
spec = SPECS[VARIANT]
mod = importlib.import_module(spec["module"])
VigenereSolver = mod.VigenereSolver

CKPT_NAME = spec["ckpt"] if not spec["two_pass"] else f"{spec['ckpt']}_s{SEED}"
pl.seed_everything(SEED, workers=True)
print(f"variant={VARIANT}  seed={SEED}  checkpoints -> {CKPT_NAME}")


In [ ]:
from models.vigenere.data_module import VigenereDataModule

MIN_K_LEN, MAX_K_LEN, BATCH_SIZE = 3, 32, 128

dm = VigenereDataModule(
    text_path=str(config.data_file("final_dataset_shuffled.txt")),
    alphabet=config.ALPHABET,
    batch_size=BATCH_SIZE,
    min_key_len=MIN_K_LEN,
    max_key_len=MAX_K_LEN,
)
dm.setup()

model = VigenereSolver(
    vocab_size=dm.dataset.crypto_vocab_size,
    min_key_len=MIN_K_LEN, max_key_len=MAX_K_LEN,
    d_model=256, nhead=8, num_layers=6, lr=1e-3,
)

n = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"parameters: {n:,}")

if spec["two_pass"]:
    assert (model.CORRECT_KEY_PROB, model.WRONG_KEY_PROB, model.KEYLEN_WEIGHT) == (0.40, 0.20, 1.00), \
        "training regime differs from the full model; the ablation would not be controlled"


In [ ]:
MAX_EPOCHS = 230
ckpt_dir, log_dir = config.model_paths(CKPT_NAME)

callbacks = [ModelCheckpoint(
    dirpath=str(ckpt_dir),
    filename=VARIANT + "-best-acc-{epoch:02d}-{val_acc:.3f}",
    save_top_k=2, save_last=True, monitor="val_acc", mode="max",
)]

if spec["two_pass"]:
    callbacks.append(ModelCheckpoint(
        dirpath=str(ckpt_dir),
        filename=VARIANT + "-best-keylen-{epoch:02d}-{val_keylen_acc:.3f}-{val_acc:.3f}",
        save_top_k=2, monitor="val_keylen_acc", mode="max",
    ))
    monitor = "val_keylen_acc"
else:
    monitor = "val_acc"

callbacks.append(EarlyStopping(monitor=monitor, patience=40, mode="max", strict=False))

trainer = pl.Trainer(
    accelerator="auto", devices=1,
    max_epochs=MAX_EPOCHS, min_epochs=80,
    callbacks=callbacks,
    logger=TensorBoardLogger(str(log_dir), name=CKPT_NAME),
    gradient_clip_val=1.0, log_every_n_steps=10, precision="32-true",
)

trainer.fit(model, datamodule=dm)


In [ ]:
import numpy as np
from models.vigenere import evaluation as ev

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
best = sorted((config.CHECKPOINT_DIR / CKPT_NAME).glob("*best-acc*.ckpt"))[-1]
m = VigenereSolver.load_from_checkpoint(str(best), map_location=DEVICE)
m.to(DEVICE).eval(); m.freeze()
print("checkpoint:", best.name)

with open(config.data_file("validation_set.txt"), encoding="utf-8") as f:
    validation_text = f.read()

TEST_LENGTHS  = [128, 256, 400, 512]
TEST_KEY_LENS = list(range(MIN_K_LEN, MAX_K_LEN + 1))

grid = ev.run_neural_eval(m, validation_text, TEST_LENGTHS, TEST_KEY_LENS,
                          config.ALPHABET, num_samples=500, device=DEVICE)

print(f"\ncharacter accuracy - {VARIANT}")
print(f"{'k':>3} | " + " ".join(f"{L:>7}" for L in TEST_LENGTHS))
for k in TEST_KEY_LENS:
    print(f"{k:>3} | " + " ".join(f"{grid[k][L]['char_acc']:>7.1f}" for L in TEST_LENGTHS))
print(f"\nmean: {np.mean([grid[k][L]['char_acc'] for k in TEST_KEY_LENS for L in TEST_LENGTHS]):.1f}%")
